# Notebook 03: Hotspot Detection using DBSCAN
**Project:** Parking-Induced Congestion AI System

This notebook:
- Loads `parking_violations.csv`
- Applies DBSCAN clustering using haversine distance on lat/lon
- Labels each record with its cluster ID
- Creates a cluster summary with violation counts, peak hours, severity levels
- Saves `parking_violations_with_clusters.csv` and `hotspot_clusters.csv`

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
import os
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported.')

Libraries imported.


## 2. Define Paths and DBSCAN Parameters

In [2]:
PARKING_CSV         = '../data/processed/parking_violations.csv'
CLUSTERED_CSV       = '../data/processed/parking_violations_with_clusters.csv'
HOTSPOT_CSV         = '../data/processed/hotspot_clusters.csv'

# DBSCAN parameters
# eps_meters: radius in meters to consider two points neighbors
# Haversine expects radians, so we convert: radians = meters / earth_radius_meters
EPS_METERS    = 200        # ~200m cluster radius (tune between 100-300m)
MIN_SAMPLES   = 5          # minimum points to form a dense region
EARTH_RADIUS  = 6371000.0  # meters

print(f'DBSCAN eps: {EPS_METERS}m, min_samples: {MIN_SAMPLES}')

DBSCAN eps: 200m, min_samples: 5


## 3. Load Parking Violations

In [3]:
df = pd.read_csv(PARKING_CSV, low_memory=False)
if 'datetime' in df.columns:
    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')

print(f'Loaded: {df.shape}')
display(df.head())

Loaded: (298450, 31)


,id,latitude,longitude,location_or_junction,vehicle_type,vehicle_type.1,description,violation_type,offence_code,created_datetime,...,updated_vehicle_type,validation_status,validation_timestamp,datetime,hour,day,month,day_of_week,weekday_name,is_weekend
0,FKID000000,12.925557,77.618665,"18Th Main Road, Block 2, Koramangala, Bengalur...",FKN00GL0000,CAR,NaN,"[""Wrong Parking"",""Parking Near Road Crossing""]","[112,104]",2023-11-20 00:28:46+00,...,MAXI-CAB,approved,2023-11-30 03:08:24.818+00,2023-11-20 00:28:46+00:00,0.0,20.0,11.0,0.0,Monday,0
1,FKID000001,12.905463,77.700778,"Sarjapura Main Road, The Grove, Janatha Colony...",FKN00GL0001,CAR,NaN,"[""No Parking""]",[113],2023-11-24 22:46:46+00,...,NaN,NaN,NaN,2023-11-24 22:46:46+00:00,22.0,24.0,11.0,4.0,Friday,0
2,FKID000002,12.925449,77.618504,"Koramangala 2Nd Block, Kormangala West, Bengal...",FKN00GL0002,CAR,NaN,"[""Wrong Parking"",""Parking In A Main Road""]","[112,107]",2023-11-20 00:27:46+00,...,MAXI-CAB,approved,2023-11-30 03:08:56.998+00,2023-11-20 00:27:46+00:00,0.0,20.0,11.0,0.0,Monday,0
3,FKID000003,12.956521,77.518618,"6Th Cross Road, Manasa Layout, Nagarbhavi, Ben...",FKN00GL0003,SCOOTER,NaN,"[""No Parking""]",[113],2023-11-16 06:47:46+00,...,SCOOTER,approved,2023-11-18 23:35:12.428+00,2023-11-16 06:47:46+00:00,6.0,16.0,11.0,3.0,Thursday,0
4,FKID000004,12.977767,77.580545,"Kalidasa Road, Gandhinagar, Nehru Nagar, Benga...",FKN00GL0004,TANKER,NaN,"[""No Parking""]",[113],2023-11-22 04:56:46+00,...,TANKER,approved,2023-11-30 03:11:32.796+00,2023-11-22 04:56:46+00:00,4.0,22.0,11.0,2.0,Wednesday,0


## 4. Validate Coordinates

In [4]:
if 'latitude' not in df.columns or 'longitude' not in df.columns:
    raise ValueError('latitude / longitude columns are required for DBSCAN. Run notebook 01 first.')

df['latitude']  = pd.to_numeric(df['latitude'],  errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

before = len(df)
df = df.dropna(subset=['latitude', 'longitude'])
print(f'Records with valid coordinates: {len(df)} (dropped {before - len(df)})')

print(f'Lat range : {df["latitude"].min():.4f}  to  {df["latitude"].max():.4f}')
print(f'Lon range : {df["longitude"].min():.4f}  to  {df["longitude"].max():.4f}')

Records with valid coordinates: 298450 (dropped 0)
Lat range : 12.8027  to  13.2937
Lon range : 77.4426  to  77.7717


## 5. Run DBSCAN Clustering (Haversine)

In [5]:
# Convert lat/lon to radians for haversine metric
coords_rad = np.radians(df[['latitude', 'longitude']].values)

eps_rad = EPS_METERS / EARTH_RADIUS
print(f'eps in radians: {eps_rad:.8f}')

dbscan = DBSCAN(
    eps=eps_rad,
    min_samples=MIN_SAMPLES,
    algorithm='ball_tree',
    metric='haversine',
    n_jobs=-1
)

print('Running DBSCAN... (may take a moment for large datasets)')
labels = dbscan.fit_predict(coords_rad)

df['cluster_id'] = labels

n_clusters    = len(set(labels)) - (1 if -1 in labels else 0)
n_noise       = (labels == -1).sum()
n_clustered   = (labels != -1).sum()

print(f'\nDBSCAN Results:')
print(f'  Total clusters found : {n_clusters}')
print(f'  Noise points (-1)    : {n_noise:,} ({n_noise/len(df)*100:.1f}%)')
print(f'  Clustered points     : {n_clustered:,} ({n_clustered/len(df)*100:.1f}%)')

eps in radians: 0.00003139
Running DBSCAN... (may take a moment for large datasets)

DBSCAN Results:
  Total clusters found : 349
  Noise points (-1)    : 756 (0.3%)
  Clustered points     : 297,694 (99.7%)


## 6. Save Full Clustered Dataset (includes noise as -1)

In [6]:
df.to_csv(CLUSTERED_CSV, index=False)
print(f'Clustered data saved: {CLUSTERED_CSV}')
print(f'Shape: {df.shape}')

Clustered data saved: ../data/processed/parking_violations_with_clusters.csv
Shape: (298450, 32)


## 7. Build Cluster Summary (Exclude Noise)

In [7]:
def safe_mode(series):
    """Return mode value or NaN if series is empty/all-null."""
    s = series.dropna()
    if len(s) == 0:
        return np.nan
    return s.mode().iloc[0]

df_clusters = df[df['cluster_id'] != -1].copy()

print(f'Building summary for {df_clusters["cluster_id"].nunique()} clusters...')

agg_dict = {
    'latitude':  'mean',
    'longitude': 'mean',
}
# Conditional aggregations
for col, canon in [
    ('violation_type',      'most_common_violation_type'),
    ('police_station',      'most_common_police_station'),
    ('location_or_junction','most_common_location_or_junction'),
]:
    if col in df_clusters.columns:
        agg_dict[col] = safe_mode

if 'hour' in df_clusters.columns:
    agg_dict['hour'] = safe_mode

cluster_summary = df_clusters.groupby('cluster_id').agg(
    violation_count=('latitude', 'count'),
    center_latitude=('latitude', 'mean'),
    center_longitude=('longitude', 'mean'),
).reset_index()

# Add mode columns separately (more control)
for col, canon in [
    ('violation_type',      'most_common_violation_type'),
    ('police_station',      'most_common_police_station'),
    ('location_or_junction','most_common_location_or_junction'),
]:
    if col in df_clusters.columns:
        mode_ser = df_clusters.groupby('cluster_id')[col].agg(safe_mode).reset_index()
        mode_ser.columns = ['cluster_id', canon]
        cluster_summary = cluster_summary.merge(mode_ser, on='cluster_id', how='left')

if 'hour' in df_clusters.columns:
    peak_hour = df_clusters.groupby('cluster_id')['hour'].agg(safe_mode).reset_index()
    peak_hour.columns = ['cluster_id', 'peak_hour']
    cluster_summary = cluster_summary.merge(peak_hour, on='cluster_id', how='left')

if 'datetime' in df_clusters.columns:
    df_clusters['_date'] = pd.to_datetime(df_clusters['datetime'], errors='coerce').dt.date
    days_active = df_clusters.groupby('cluster_id')['_date'].nunique().reset_index()
    days_active.columns = ['cluster_id', 'unique_days_count']
    cluster_summary = cluster_summary.merge(days_active, on='cluster_id', how='left')

    if 'month' in df_clusters.columns:
        months_active = df_clusters.groupby('cluster_id')['month'].nunique().reset_index()
        months_active.columns = ['cluster_id', 'active_months_count']
        cluster_summary = cluster_summary.merge(months_active, on='cluster_id', how='left')

print(f'Cluster summary shape: {cluster_summary.shape}')
display(cluster_summary.head())

Building summary for 349 clusters...
Cluster summary shape: (349, 10)


,cluster_id,violation_count,center_latitude,center_longitude,most_common_violation_type,most_common_police_station,most_common_location_or_junction,peak_hour,unique_days_count,active_months_count
0,0,11967,12.915845,77.627327,"[""Wrong Parking""]",Hsr Layout,"20Th Main Road, Block 7, Koramangala, Bengalur...",21.0,152,6
1,1,1637,12.904944,77.682820,"[""Wrong Parking""]",Bellandur,"Amrutha College Road, Azalea, Owners Court Lay...",5.0,145,6
2,2,1250,12.957742,77.516240,"[""Wrong Parking""]",Byatarayanapura,"Gnana Bharathi Road, Manasa Layout, Nagarbhavi...",6.0,136,6
3,3,186628,12.978381,77.576195,"[""Wrong Parking""]",Upparpet,"Kamaraj Road, Sri Nagamma Devi Circle, Sivanch...",5.0,152,6
4,4,5596,13.007850,77.693117,"[""Wrong Parking""]",K.R. Pura,"Mbt Road, Devasandra Junction, Kr Puram, Benga...",19.0,152,6


## 8. Assign Severity Levels

In [8]:
def assign_severity(counts):
    """
    Assign severity based on quantiles so it adapts to dataset size.
    Low    : < 25th percentile
    Medium : 25th-50th
    High   : 50th-75th
    Critical: > 75th
    """
    q25 = counts.quantile(0.25)
    q50 = counts.quantile(0.50)
    q75 = counts.quantile(0.75)

    print(f'Severity thresholds -> Low:<{q25:.0f}, Medium:{q25:.0f}-{q50:.0f}, '
          f'High:{q50:.0f}-{q75:.0f}, Critical:>{q75:.0f}')

    def label(v):
        if v <= q25:  return 'Low'
        if v <= q50:  return 'Medium'
        if v <= q75:  return 'High'
        return 'Critical'

    return counts.apply(label)

cluster_summary['severity_level'] = assign_severity(cluster_summary['violation_count'])
print('\nSeverity distribution:')
print(cluster_summary['severity_level'].value_counts())

Severity thresholds -> Low:<8, Medium:8-20, High:20-90, Critical:>90

Severity distribution:
severity_level
Low         89
Medium      88
High        86
Critical    86
Name: count, dtype: int64


## 9. Save Hotspot Summary

In [9]:
cluster_summary = cluster_summary.sort_values('violation_count', ascending=False).reset_index(drop=True)
cluster_summary.to_csv(HOTSPOT_CSV, index=False)
print(f'Hotspot clusters saved: {HOTSPOT_CSV}')
print(f'Shape: {cluster_summary.shape}')

Hotspot clusters saved: ../data/processed/hotspot_clusters.csv
Shape: (349, 11)


## 10. Top 10 Clusters Summary

In [10]:
print(f'\nTotal clusters     : {n_clusters}')
print(f'Noise points (-1)  : {n_noise:,}')
print('\nTop 10 clusters by violation count:')
display_cols = [c for c in [
    'cluster_id','violation_count','center_latitude','center_longitude',
    'most_common_violation_type','most_common_police_station',
    'most_common_location_or_junction','peak_hour','severity_level'
] if c in cluster_summary.columns]
display(cluster_summary[display_cols].head(10))


Total clusters     : 349
Noise points (-1)  : 756

Top 10 clusters by violation count:


,cluster_id,violation_count,center_latitude,center_longitude,most_common_violation_type,most_common_police_station,most_common_location_or_junction,peak_hour,severity_level
0,3,186628,12.978381,77.576195,"[""Wrong Parking""]",Upparpet,"Kamaraj Road, Sri Nagamma Devi Circle, Sivanch...",5.0,Critical
1,6,20204,12.936777,77.691050,"[""No Parking""]",Hal Old Airport,"New Horizon College Road, New Horizon College ...",22.0,Critical
2,0,11967,12.915845,77.627327,"[""Wrong Parking""]",Hsr Layout,"20Th Main Road, Block 7, Koramangala, Bengalur...",21.0,Critical
3,21,7006,13.039466,77.589575,"[""No Parking""]",Hebbala,"Bellary Road, Vinayaka Nagar, Hebbal, Bengalur...",5.0,Critical
4,28,6226,13.070997,77.588444,"[""Wrong Parking""]",Kodigehalli,"Sahakar Nagar Road, Fortuna Acacia, Byatarayan...",6.0,Critical
5,10,6028,12.966517,77.645642,"[""No Parking""]",Jeevanbheemanagar,"Domlur Inner Ring Road, Hal Stage 2, Indiranag...",2.0,Critical
6,13,5871,12.997640,77.679415,"[""No Parking""]",Mahadevapura,"Outer Ring Road, Venkatappa Colony, Mahadevapu...",4.0,Critical
7,4,5596,13.007850,77.693117,"[""Wrong Parking""]",K.R. Pura,"Mbt Road, Devasandra Junction, Kr Puram, Benga...",19.0,Critical
8,14,5145,13.185562,77.680493,"[""No Parking""]",Chikkajala,"Unnamed Road, Begur Chikkanahalli, Bengaluru, ...",23.0,Critical
9,35,4754,12.965592,77.715621,"[""Wrong Parking""]",Hal Old Airport,"Itpl Main Road, Brigade Tech Gardens, Brookefi...",23.0,Critical
